In [ ]:
from scipy.stats import pointbiserialr, chi2_contingency
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.size":        11,
    "axes.titlesize":   11,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "legend.fontsize":  9,
})

In [ ]:
df = pd.read_csv('../data/HAM10000_metadata_cleaned.csv')

CLASSES = sorted(df['dx'].unique())
CLASS_NAMES = {
    'akiec': 'Actinic Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'bkl':   'Benign Keratosis',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevus',
    'vasc':  'Vascular Lesion'
}
COLORS = {
    'akiec': '#E57E46', 'bcc': '#4A579E', 'bkl': '#83BE59',
    'df':    '#F46ACD', 'mel': '#885646', 'nv':  '#6692EA',
    'vasc':  '#A68898'
}
bins = np.arange(0, 95, 5)

fig, axes = plt.subplots(len(CLASSES), 3, figsize=(20, 5 * len(CLASSES)), dpi=150)

for row, cls in enumerate(CLASSES):
    df[f'is_{cls}'] = (df['dx'] == cls).astype(int)
    color    = COLORS[cls]
    cls_name = CLASS_NAMES[cls]

    r_a, p_a          = pointbiserialr(df[f'is_{cls}'], df['age'])
    chi2_s, p_s, _, _ = chi2_contingency(pd.crosstab(df['sex'],          df[f'is_{cls}']))
    chi2_l, p_l, _, _ = chi2_contingency(pd.crosstab(df['localization'], df[f'is_{cls}']))

    cls_age  = df[df['dx'] == cls]['age']
    rest_age = df[df['dx'] != cls]['age']

    ax = axes[row, 0]
    ax.hist(rest_age, bins=bins, alpha=0.75, color='#AAAAAA', density=True)
    ax.hist(cls_age,  bins=bins, alpha=0.75, color=color,     density=True)
    ax.set_xlabel("Age")
    ax.set_ylabel("Density (%)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:.1f}%"))
    ax.legend(
        handles=[plt.Rectangle((0,0),1,1, color='#AAAAAA', alpha=0.75),
                 plt.Rectangle((0,0),1,1, color=color,     alpha=0.75),
                 plt.Rectangle((0,0),1,1, alpha=0)],
        labels=[f'rest (n={len(rest_age)})', f'{cls_name} (n={len(cls_age)})',
                f'r={r_a:.3f}, p={p_a:.2e}'],
        fontsize=8
    )
    ax.grid(True, which="major", linestyle="-", linewidth=0.5, alpha=0.5)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.5)
    ax.xaxis.set_major_locator(plt.MultipleLocator(10))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

    ax = axes[row, 1]
    sex_r = df.groupby('sex')[f'is_{cls}'].mean() * 100
    sex_c = df.groupby('sex')[f'is_{cls}'].count()
    ax.bar(sex_r.index, sex_r.values, color=['#F46ACD', '#4A579E'], alpha=0.8)
    ax.set_ylabel(f"% {cls_name}")
    ax.set_ylim(0, sex_r.max() * 1.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    for i, (k, v) in enumerate(sex_r.items()):
        ax.text(i, v + 0.3, f"{v:.1f}%\n(n={sex_c[k]})", ha='center', fontsize=8)
    ax.legend(
        handles=[plt.Rectangle((0,0),1,1, alpha=0)],
        labels=[f"χ²={chi2_s:.2f}, p={p_s:.2e}"],
        handlelength=0, handletextpad=0, fontsize=8
    )
    ax.grid(True, which="major", linestyle="-", linewidth=0.5, alpha=0.5, axis='y')
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.5, axis='y')
    ax.yaxis.set_major_locator(plt.MultipleLocator(5))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))

    ax = axes[row, 2]
    loc_r = df.groupby('localization')[f'is_{cls}'].mean() * 100
    loc_c = df.groupby('localization')[f'is_{cls}'].count()
    loc_r = loc_r.sort_values(ascending=True)
    ax.barh(loc_r.index, loc_r.values, color=color, alpha=0.8)
    ax.set_xlabel(f"% {cls_name}")
    for i, (loc, v) in enumerate(loc_r.items()):
        ax.text(v + 0.2, i, f"{v:.1f}% (n={loc_c[loc]})", va='center', fontsize=7)
    ax.legend(
        handles=[plt.Rectangle((0,0),1,1, alpha=0)],
        labels=[f"χ²={chi2_l:.2f}, p={p_l:.2e}"],
        handlelength=0, handletextpad=0, fontsize=8
    )
    ax.grid(True, which="major", linestyle="-", linewidth=0.5, alpha=0.5, axis='x')
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.5, axis='x')
    ax.xaxis.set_major_locator(plt.MultipleLocator(10))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

plt.tight_layout()
plt.show()